# Experiment 2 — W&B Bayesian hyperparameter sweep

This experiment searches architecture and training hyperparameters for the lowest validation conditional flow-matching loss. Because this is an unconditional generative model, the optimization metric is `loss/val_best`, not classification accuracy.

A literal Cartesian product of every value would require hundreds of thousands of runs. Bayesian search plus Hyperband explores the same parameter space within a practical budget. Every run uses the same number of training steps so validation losses remain comparable.

In [ ]:
import gc
import os
from pathlib import Path

import torch
import wandb

from datasets.mnist import MNISTSampler
from models.flow import FlowModel
from models.time_embedding import SinusoidalTimeEmbeddings
from models.unet import UNet
from training.path import GaussianConditionalProbabilityPath, LinearAlpha, LinearBeta
from training.trainer import FlowTrainer, model_size_b

PROJECT = "mnist-flow-matching"
SWEEP_RUNS = 30
SEARCH_STEPS = 1_000
FINAL_STEPS = 5_000
VAL_EVERY = 50
VAL_BATCHES = 8

persistent_root = Path(
    os.getenv("DIFFUSION_DATA_ROOT", "/workspace-global/Diffusion-data")
)
data_root = persistent_root / "datasets" / "mnist"
checkpoints_dir = persistent_root / "checkpoints" / "experiment_2"
# W&B already syncs logs to its service; use fast local disk for its many small files.
wandb_dir = Path("/workspace/wandb")
for path in (data_root, checkpoints_dir, wandb_dir):
    path.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("Experiment 2 is intended to run on a CUDA GPU")
torch.set_float32_matmul_precision("high")
print("device", device, torch.cuda.get_device_name(0))

# Download files persist globally; transformed tensors are cached in GPU memory
# once and reused by all sweep runs in this agent process.
train_data = MNISTSampler(root=str(data_root), split="train").to(device)
val_data = MNISTSampler(root=str(data_root), split="val").to(device)
print("cached images", len(train_data.images), len(val_data.images))

In [ ]:
sweep_config = {
    "name": "experiment-2-model-and-training-search",
    "method": "bayes",
    "metric": {"name": "loss/val_best", "goal": "minimize"},
    "early_terminate": {"type": "hyperband", "min_iter": 200},
    "parameters": {
        # Model architecture. Input channels and image size are fixed by MNIST.
        "start_dim": {"values": [32, 64, 96]},
        "dim_mults": {"values": ["1,2", "1,2,4"]},
        "residual_blocks_per_group": {"values": [1, 2]},
        "group_norm_num_groups": {"values": [8, 16]},
        "n_heads": {"values": [2, 4, 8]},
        "time_embedding_dim": {"values": [64, 128, 256]},
        "conditioning_dim": {"values": [128, 256, 512]},
        "d_ff": {"values": [128, 256, 512]},
        "activation": {"values": ["gelu", "silu"]},
        "attn_dropout": {"values": [0.0, 0.1, 0.2]},
        "ffn_dropout": {"values": [0.0, 0.1, 0.2]},
        "proj_dropout": {"values": [0.0, 0.1, 0.2]},
        # Training hyperparameters. The step budget and validation protocol stay
        # fixed so the sweep compares runs fairly.
        "optimizer": {"values": ["adam", "adamw"]},
        "learning_rate": {
            "distribution": "log_uniform_values",
            "min": 1e-4,
            "max": 3e-3,
        },
        "weight_decay": {"values": [0.0, 1e-5, 1e-4, 1e-3]},
        "max_grad_norm": {"values": [0.5, 1.0, 5.0]},
        "batch_size": {"values": [32, 64, 128]},
    },
}

sweep_config

In [ ]:
def train_config(run, num_steps: int) -> None:
    config = run.config
    torch.manual_seed(42)

    dim_mults = tuple(int(value) for value in config.dim_mults.split(","))
    time_embed = SinusoidalTimeEmbeddings(
        time_emb_dim=config.time_embedding_dim,
        scaled_time_emb_dim=config.conditioning_dim,
    )
    unet = UNet(
        in_channels=1,
        start_dim=config.start_dim,
        dim_mults=dim_mults,
        residual_blocks_per_group=config.residual_blocks_per_group,
        group_norm_num_groups=config.group_norm_num_groups,
        time_emb_dim=config.conditioning_dim,
        n_heads=config.n_heads,
        d_ff=config.d_ff,
        attn_dropout=config.attn_dropout,
        ffn_dropout=config.ffn_dropout,
        proj_dropout=config.proj_dropout,
        activation=config.activation,
    )
    model = FlowModel(unet=unet, time_embed=time_embed).to(device)

    def make_path(data: MNISTSampler) -> GaussianConditionalProbabilityPath:
        return GaussianConditionalProbabilityPath(
            p_data=data,
            p_simple_shape=[1, 32, 32],
            alpha=LinearAlpha(),
            beta=LinearBeta(),
        ).to(device)

    train_path = make_path(train_data)
    val_path = make_path(val_data)
    trainer = FlowTrainer(path=train_path, model=model, val_path=val_path)
    checkpoint_path = checkpoints_dir / f"{run.id}-{num_steps}-steps.pt"

    run.summary["model/parameters"] = sum(p.numel() for p in model.parameters())
    run.summary["model/size_mib"] = model_size_b(model) / 1024**2

    try:
        history = trainer.train(
            num_steps=num_steps,
            device=device,
            lr=config.learning_rate,
            optimizer_name=config.optimizer,
            weight_decay=config.weight_decay,
            max_grad_norm=config.max_grad_norm,
            batch_size=config.batch_size,
            ckpt_path=checkpoint_path,
            checkpoint_every=num_steps,
            val_every=VAL_EVERY,
            val_batches=VAL_BATCHES,
            plot_every=0,
            show_plots=False,
            wandb_run=run,
        )
        run.summary["loss/train_best"] = history["train"].min().item()
        run.summary["loss/val_best"] = history["val"].min().item()
        run.summary["checkpoint"] = str(checkpoint_path)
        run.summary["training_steps"] = num_steps
    finally:
        del trainer, model, train_path, val_path
        gc.collect()
        torch.cuda.empty_cache()


def run_trial() -> None:
    with wandb.init(project=PROJECT, dir=str(wandb_dir)) as run:
        train_config(run, SEARCH_STEPS)

In [ ]:
# This prompts once if the pod does not already have a W&B API key.
wandb.login()
sweep_id = wandb.sweep(sweep=sweep_config, project=PROJECT)
(persistent_root / "experiment_2_sweep_id.txt").write_text(sweep_id)
print("sweep_id", sweep_id)

In [ ]:
# Run more agents (on this or other pods) with the same sweep_id to parallelize.
wandb.agent(sweep_id, function=run_trial, count=SWEEP_RUNS, project=PROJECT)

In [ ]:
api = wandb.Api()
sweep = api.sweep(f"{api.default_entity}/{PROJECT}/{sweep_id}")
# Only rank equal-budget search trials. This excludes later winner retraining runs.
search_runs = [
    run for run in sweep.runs
    if run.summary.get("training_steps") == SEARCH_STEPS
    and run.summary.get("loss/val_best") is not None
]
best_run = min(search_runs, key=lambda run: run.summary["loss/val_best"])

print("best run:", best_run.name)
print("best validation loss:", best_run.summary.get("loss/val_best"))
print("best training loss:", best_run.summary.get("loss/train_best"))
print("checkpoint:", best_run.summary.get("checkpoint"))
print("config:")
for key, value in sorted(best_run.config.items()):
    print(f"  {key}: {value}")

## Retrain the winner fairly

The sweep uses 1,000-step trials to control cost. Run the following cell after the sweep to train its winner for 5,000 steps, matching Experiment 1. Compare this run's `loss/train_best` and `loss/val_best` against the Experiment 1 run in the same W&B project.

In [ ]:
winner_config = dict(best_run.config)
wandb.teardown()
# wandb.agent leaves the last sweep run in the process environment. Clear it
# so this becomes a fresh, ordinary run with the winning configuration.
for key in ("WANDB_RUN_ID", "WANDB_SWEEP_ID", "WANDB_SWEEP_PARAM_PATH"):
    os.environ.pop(key, None)

with wandb.init(
    project=PROJECT,
    dir=str(wandb_dir),
    name="experiment-2-winner-5000-steps",
    group=sweep_id,
    tags=["experiment-2", "best-retrain"],
    config=winner_config,
) as final_run:
    train_config(final_run, FINAL_STEPS)